In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_product_master = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load("abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Bronze.Lakehouse/Files/bronze/product_master/product_master.csv")
)

print("Rows:", df_product_master.count())
print("Columns:", len(df_product_master.columns))

display(df_product_master)

StatementMeta(, e3c4317c-e3ef-42b7-8174-6e4b1670061a, 3, Finished, Available, Finished, False)

Rows: 50
Columns: 7


SynapseWidget(Synapse.DataFrame, 31bd1dd4-37ff-471b-bc94-6886fe6f5aa6)

In [2]:
duplicate_product_ids = (
    df_product_master
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate product IDs:", duplicate_product_ids.count())

display(duplicate_product_ids)

StatementMeta(, e3c4317c-e3ef-42b7-8174-6e4b1670061a, 4, Finished, Available, Finished, False)

Duplicate product IDs: 0


SynapseWidget(Synapse.DataFrame, 57ba74b8-8123-4e7a-9860-acd6f2dfb742)

In [3]:
null_counts = df_product_master.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_product_master.columns
])

display(null_counts)

StatementMeta(, e3c4317c-e3ef-42b7-8174-6e4b1670061a, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 738fa248-151f-4c01-8c55-f2a35f665557)

In [4]:
display(
    df_product_master
    .groupBy("active_flag")
    .count()
    .orderBy("active_flag")
)

StatementMeta(, e3c4317c-e3ef-42b7-8174-6e4b1670061a, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a996573e-ca2b-4a5e-a8d4-297d077f23f3)

In [5]:
from pyspark.sql import functions as F

df_product_dim = (
    df_product_master
    .withColumn(
        "product_id",
        F.upper(F.trim(F.col("product_id")))
    )
    .withColumn(
        "product_name",
        F.trim(F.col("product_name"))
    )
    .withColumn(
        "product_category",
        F.trim(F.col("product_category"))
    )
    .withColumn(
        "product_family",
        F.trim(F.col("product_family"))
    )
    .withColumn(
        "business_unit",
        F.trim(F.col("business_unit"))
    )
    .withColumn(
        "active_flag",
        F.trim(F.col("active_flag")))
)

display(df_product_dim)

StatementMeta(, e3c4317c-e3ef-42b7-8174-6e4b1670061a, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dfc6f0dd-f8d6-4add-99ee-03361644e87e)

In [6]:
(
    df_product_dim.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dim_product")
)

StatementMeta(, e3c4317c-e3ef-42b7-8174-6e4b1670061a, 8, Finished, Available, Finished, False)

In [7]:
df_silver_product = spark.read.table("dim_product")

print(
    "Silver dim_product rows:",
    df_silver_product.count()
)

print(
    "Silver dim_product columns:",
    len(df_silver_product.columns)
)

display(df_silver_product)

StatementMeta(, e3c4317c-e3ef-42b7-8174-6e4b1670061a, 9, Finished, Available, Finished, False)

Silver dim_product rows: 50
Silver dim_product columns: 7


SynapseWidget(Synapse.DataFrame, 831ac539-26c2-4dae-8b44-f91905f44fba)